This is my second model for SS2 Sementic Search Model applied to the same 3 WANDS data sets as the pervious BM25 model . This Model applies Sentence Trasformers to the search queries and product infomation database , this infomation is than embbeded to numerical embeddings.Than all products are compared and ranked to customer searches 

In [3]:
# Import the libraries needed

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

print("Libraries loaded successfully")

Libraries loaded successfully


In [30]:

text_columns = [
    'product_name',
    'product_class',
    'category hierarchy',
    'product_description',
    'product_features'
]


for column in text_columns:
    product_df[column] = product_df[column].fillna('').astype(str)


product_df['search_document'] = (
    product_df['product_name'] + ' ' +
    product_df['product_class'] + ' ' +
    product_df['category hierarchy'] + ' ' +
    product_df['product_description'] + ' ' +
    product_df['product_features']
)

print("Search documents created:", len(product_df))

product_df[['product_name', 'search_document']].head()

Search documents created: 42994


,product_name,search_document
0,solid wood platform bed,solid wood platform bed Beds Furniture / Bedro...
1,all-clad 7 qt . slow cooker,all-clad 7 qt . slow cooker Slow Cookers Kitch...
2,all-clad electrics 6.5 qt . slow cooker,all-clad electrics 6.5 qt . slow cooker Slow C...
3,all-clad all professional tools pizza cutter,all-clad all professional tools pizza cutter S...
4,baldwin prestige alcott passage knob with roun...,baldwin prestige alcott passage knob with roun...


In [8]:
print('search_document' in product_df.columns)

True


All the datasets have been loaded and prepared , missing values handeled and most importantly product fields were combined to a single ' search document' now for the model we need to install Sentence Trasformer to convert the product  infomation and search queries to numerical embeddings that create the semantic meaning 

In [31]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Semantic model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic model loaded successfully!


In [11]:
# Create product embeddings

product_texts = product_df['search_document'].tolist()

product_embeddings = model.encode(
    product_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Product embeddings created successfully!")
print("Embedding shape:", product_embeddings.shape)

Batches:   0%|          | 0/672 [00:00<?, ?it/s]

Product embeddings created successfully!
Embedding shape: (42994, 384)


In [20]:
np.save("product_embeddings.npy", product_embeddings)

print("Product embeddings saved successfully!")

Product embeddings saved successfully!


In [21]:
product_embeddings = np.load("product_embeddings.npy")

print("Product embeddings loaded successfully!")
print("Embedding shape:", product_embeddings.shape)

Product embeddings loaded successfully!
Embedding shape: (42994, 384)


There is where the Test Query of Boho bed frame that I began in BM25 is carried over and the expirement is continued to be able to compare the findings after i have installed the Model . The BM25 model did well overall but struggled with ranking , now we comapre the Semantic Model using the same example. 

In [23]:
test_query = "boho bed frame"


query_embedding = model.encode(
    test_query,
    normalize_embeddings=True
)


semantic_scores = product_embeddings @ query_embedding


top_10 = np.argsort(semantic_scores)[::-1][:10]


product_df.iloc[top_10][['product_id', 'product_name']]

,product_id,product_name
27655,27655,boho chic platform bed
18005,18005,borah upholstered low profile sleigh bed
23034,23034,bodhi twin platform bed
30942,30942,bomke twin platform bed
33068,33068,bordiuk twin platform bed
19807,19807,boswell upholstered platform bed
26156,26156,platform bed
15675,15675,bolick full platform bed with trundle
27654,27654,boho chic 2 drawer nightstand
39844,39844,dorinda king solid wood platform bed


The Semantic model has ranked our boho chic platform bed as number one for the search query of boho bed frame 

In [18]:
label_df[
    (label_df['query_id'] == 125) &
    (label_df['product_id'] == 27655)
]

,id,query_id,product_id,label
17742,17742,125,27655,Exact


The Semantic model has ranked our boho chic platform bed as number one for the search query of boho bed frame . And the WANDS human relevence label comfirm clear improvement with rank as 1 and the human label . This demonstates benefit of the semantic embeddings of the products and the customer query when diffrent words are used for the same type of product. 

Now , to look at the evaluation metrics of the model , I will keep the consistency of the BM25 model and only look at the first 10 prducts .

In [26]:
precision_scores = []

for _, row in query_df.iterrows():

   
    query_embedding = model.encode(
        row['query'],
        normalize_embeddings=True
    )

    
    scores = product_embeddings @ query_embedding


    top_10 = np.argsort(scores)[::-1][:10]
    top_products = product_df.iloc[top_10][['product_id']].copy()

    
    top_products['query_id'] = row['query_id']

    
    judged = top_products.merge(
        label_df[['query_id', 'product_id', 'label']],
        on=['query_id', 'product_id'],
        how='inner'
    )


    if len(judged) > 0:
        relevant = judged['label'].isin(['Exact', 'Partial']).sum()
        precision_scores.append(relevant / len(judged))



semantic_precision_at_10 = np.mean(precision_scores)

print("Semantic Judged Precision@10:", round(semantic_precision_at_10, 4))
print("Semantic Judged Precision@10 (%):", round(semantic_precision_at_10 * 100, 2))

Semantic Judged Precision@10: 0.9565
Semantic Judged Precision@10 (%): 95.65


In [27]:
recall_scores = []

for _, row in query_df.iterrows():

    query_embedding = model.encode(
        row['query'],
        normalize_embeddings=True
    )

    scores = product_embeddings @ query_embedding
    top_10 = np.argsort(scores)[::-1][:10]

    retrieved_ids = set(product_df.iloc[top_10]['product_id'])

    relevant_ids = set(
        label_df[
            (label_df['query_id'] == row['query_id']) &
            (label_df['label'].isin(['Exact', 'Partial']))
        ]['product_id']
    )

    if len(relevant_ids) > 0:
        retrieved_relevant = len(retrieved_ids & relevant_ids)
        recall_scores.append(retrieved_relevant / len(relevant_ids))

semantic_recall_at_10 = np.mean(recall_scores)

print("Semantic Recall@10:", round(semantic_recall_at_10, 4))
print("Semantic Recall@10 (%):", round(semantic_recall_at_10 * 100, 2))

Semantic Recall@10: 0.0571
Semantic Recall@10 (%): 5.71


In [28]:
from sklearn.metrics import ndcg_score

ndcg_scores = []

for _, row in query_df.iterrows():

    query_embedding = model.encode(
        row['query'],
        normalize_embeddings=True
    )

    scores = product_embeddings @ query_embedding
    top_10 = np.argsort(scores)[::-1][:10]

    top_products = product_df.iloc[top_10][['product_id']].copy()
    top_products['query_id'] = row['query_id']

    results = top_products.merge(
        label_df[['query_id', 'product_id', 'label']],
        on=['query_id', 'product_id'],
        how='left'
    )

    
    relevance = results['label'].map({
        'Exact': 2,
        'Partial': 1,
        'Irrelevant': 0
    }).fillna(0).to_numpy()

    if relevance.sum() > 0:
        ideal = np.sort(relevance)[::-1]
        ndcg_scores.append(
            ndcg_score([ideal], [relevance], k=10)
        )

semantic_ndcg_at_10 = np.mean(ndcg_scores)

print("Semantic NDCG@10:", round(semantic_ndcg_at_10, 4))
print("Semantic NDCG@10 (%):", round(semantic_ndcg_at_10 * 100, 2))

Semantic NDCG@10: 0.8846
Semantic NDCG@10 (%): 88.46
